# FDSA Real Benchmark — Drift-Filtered Decoding vs Baseline

**Purpose:** Test the Actualizer Engine / FDSA drift-filtering method against standard decoding
on a *real* closed-book QA dataset with checkable ground truth, using a real pretrained model
(Flax T5) on TPU — replacing the synthetic 500-token toy-grammar benchmark.

**Honesty rule for this notebook:** every result printed at the end reports both baseline and
FDSA-filtered numbers side by side. If FDSA does not help, or hurts, that is reported as-is.
Do not report only the metric that looks favorable.

**Before running for real:** Cell 6 below (`fdsa_logits_processor`) is a PLACEHOLDER, not your
real Actualizer Engine / FDSA implementation. Replace its body with your actual drift-tensor
filtering logic (from `Actualizer_Engine.py` / `fdsa_pruner.py`) before treating any comparison
here as a valid test of your method.


## 1. Environment setup — run this first in Colab (Runtime > Change runtime type > TPU)

In [ ]:
# Verify TPU is visible
import jax
print("JAX devices:", jax.devices())
print("Device count:", jax.device_count())
assert jax.device_count() > 0, "No TPU/accelerator visible — check Runtime > Change runtime type"


/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:88: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


JAX devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]
Device count: 1


In [ ]:
# Pin transformers and its internal dependencies to bypass metadata conflicts in Python 3.12
!pip install -q --upgrade "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
!pip install -q --upgrade flax datasets evaluate sentencepiece
!pip install -q --no-deps transformers==4.48.2 tokenizers==0.21.0 huggingface-hub==0.24.0

import transformers
import sentencepiece
print(f"Transformers version: {transformers.__version__}")

# Diagnostic: This should show Flax classes AFTER a session restart
flax_objects = [obj for obj in dir(transformers) if 'Flax' in obj]
print(f"Available Flax objects in transformers: {flax_objects}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.0/419.0 kB 12.8 MB/s eta 0:00:00
Transformers version: 4.48.2
Available Flax objects in transformers: ['FlaxAlbertForMaskedLM', 'FlaxAlbertForMultipleChoice', 'FlaxAlbertForPreTraining', 'FlaxAlbertForQuestionAnswering', 'FlaxAlbertForSequenceClassification', 'FlaxAlbertForTokenClassification', 'FlaxAlbertModel', 'FlaxAlbertPreTrainedModel', 'FlaxAutoModel', 'FlaxAutoModelForCausalLM', 'FlaxAutoModelForImageClassification', 'FlaxAutoModelForMaskedLM', 'FlaxAutoModelForMultipleChoice', 'FlaxAutoModelForNextSentencePrediction', 'FlaxAutoModelForPreTraining', 'FlaxAutoModelForQuestionAnswering', 'FlaxAutoModelForSeq2SeqLM', 'FlaxAutoModelForSequenceClassification', 'FlaxAutoModelForSpeechSeq2Seq', 'FlaxAutoModelForTokenClassification', 'FlaxAutoModelForVision2Seq', 'FlaxBartDecoderPreTrainedModel', 'FlaxBartForCausalLM', 'FlaxBartForConditionalGeneration', 'FlaxBartForQuestionAnswering', 'FlaxBartForSequenceClassification', 'FlaxBartModel', 

In [ ]:
import transformers
import flax
import sentencepiece
from transformers import FlaxT5ForConditionalGeneration

print(f"Transformers version: {transformers.__version__}")
print(f"Flax version: {flax.__version__}")
print(f"Sentencepiece version: {sentencepiece.__version__}")
print("SUCCESS: FlaxT5ForConditionalGeneration imported correctly.")

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:88: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


Transformers version: 4.48.2
Flax version: 0.12.7
Sentencepiece version: 0.2.2
SUCCESS: FlaxT5ForConditionalGeneration imported correctly.


## 2. Load model — Flax T5 (small enough for free-tier TPU iteration)

In [ ]:
from transformers import AutoTokenizer, FlaxT5ForConditionalGeneration
import jax.numpy as jnp
import jax
import time

# Ensure we capture the TRUE original JAX clip to avoid recursion loops
import jax._src.numpy.lax_numpy as lax_numpy
_true_orig_clip = lax_numpy.clip

def patched_clip(a, a_min=None, a_max=None, **kwargs):
    # Force positional mapping to satisfy JAX's underlying implementation
    return _true_orig_clip(a, a_min, a_max)

# Apply the fix globally
jnp.clip = patched_clip

MODEL_NAME = "google/flan-t5-base"

print(f"Loading tokenizer and model: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = FlaxT5ForConditionalGeneration.from_pretrained(MODEL_NAME)

print(f"Successfully loaded {MODEL_NAME}")

Loading tokenizer and model: google/flan-t5-base...


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Successfully loaded google/flan-t5-base


## 3. Load a real dataset with ground truth

Using TriviaQA (rc.nocontext config = closed-book, no supporting doc, so the model must rely on
what it "knows" — this is exactly the condition where hallucination shows up). Small subset for
feasibility on free-tier TPU.


In [ ]:
from datasets import load_dataset

N_EXAMPLES = 100  # increase once the pipeline is confirmed working end-to-end

raw = load_dataset("trivia_qa", "rc.nocontext", split=f"validation[:{N_EXAMPLES}]")

def format_example(ex):
    question = ex["question"]
    # trivia_qa provides a list of acceptable normalized aliases as ground truth
    answers = ex["answer"]["normalized_aliases"] + [ex["answer"]["normalized_value"]]
    return {"question": question, "answers": list(set(answers))}

eval_set = [format_example(ex) for ex in raw]
print(f"Loaded {len(eval_set)} closed-book QA examples")
print("Example:", eval_set[0])


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

train-00000-of-00001.parquet:   0%|          | 0.00/55.4M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/7.34M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/1.20M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/138384 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/17944 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/17210 [00:00<?, ? examples/s]

Loaded 100 closed-book QA examples
Example: {'question': 'Who was the man behind The Chipmunks?', 'answers': ['david seville']}


## 4. Scoring — exact match / F1 against the real answer aliases

In [ ]:
import re, string
from collections import Counter

def normalize_answer(s):
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ''.join(ch for ch in s if ch not in string.punctuation)
    s = ' '.join(s.split())
    return s

def exact_match(prediction, ground_truths):
    pred_norm = normalize_answer(prediction)
    return float(any(pred_norm == normalize_answer(gt) for gt in ground_truths))

def f1_score(prediction, ground_truths):
    pred_tokens = normalize_answer(prediction).split()
    best = 0.0
    for gt in ground_truths:
        gt_tokens = normalize_answer(gt).split()
        common = Counter(pred_tokens) & Counter(gt_tokens)
        num_same = sum(common.values())
        if num_same == 0:
            continue
        precision = num_same / len(pred_tokens)
        recall = num_same / len(gt_tokens)
        f1 = 2 * precision * recall / (precision + recall)
        best = max(best, f1)
    return best


## 5. Baseline decoding — standard beam search, no filtering

This is the control condition. Run this cell as-is first to get baseline numbers.


In [ ]:
from datasets import load_dataset
import time
import re, string
from collections import Counter

# --- Utilities ---
def normalize_answer(s):
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ''.join(ch for ch in s if ch not in string.punctuation)
    s = ' '.join(s.split())
    return s

def exact_match(prediction, ground_truths):
    pred_norm = normalize_answer(prediction)
    return float(any(pred_norm == normalize_answer(gt) for gt in ground_truths))

def f1_score(prediction, ground_truths):
    pred_tokens = normalize_answer(prediction).split()
    best = 0.0
    for gt in ground_truths:
        gt_tokens = normalize_answer(gt).split()
        common = Counter(pred_tokens) & Counter(gt_tokens)
        num_same = sum(common.values())
        if num_same == 0:
            continue
        precision = num_same / len(pred_tokens)
        recall = num_same / len(gt_tokens)
        f1 = 2 * precision * recall / (precision + recall)
        best = max(best, f1)
    return best

def run_generation(questions, generate_kwargs, batch_size=8):
    predictions = []
    total_tokens = 0
    start = time.time()

    for i in range(0, len(questions), batch_size):
        batch = questions[i:i+batch_size]
        prompts = [f"trivia question: {q}" for q in batch]
        inputs = tokenizer(prompts, return_tensors="jax", padding=True, truncation=True, max_length=64)
        out = model.generate(**inputs, **generate_kwargs)
        decoded = tokenizer.batch_decode(out.sequences, skip_special_tokens=True)
        predictions.extend(decoded)
        total_tokens += out.sequences.size

    elapsed = time.time() - start
    tokens_per_sec = total_tokens / elapsed if elapsed > 0 else float("nan")
    return predictions, elapsed, tokens_per_sec

# --- Data Setup ---
N_EXAMPLES = 100
try:
    _ = eval_set
except NameError:
    print("Loading TriviaQA dataset...")
    raw = load_dataset("trivia_qa", "rc.nocontext", split=f"validation[:{N_EXAMPLES}]")
    def format_example(ex):
        answers = ex["answer"]["normalized_aliases"] + [ex["answer"]["normalized_value"]]
        return {"question": ex["question"], "answers": list(set(answers))}
    eval_set = [format_example(ex) for ex in raw]

# --- Run Baseline ---
questions = [ex["question"] for ex in eval_set]
baseline_kwargs = dict(max_new_tokens=16, num_beams=4, do_sample=False)

print("Running baseline generation...")
baseline_preds, baseline_time, baseline_tps = run_generation(questions, baseline_kwargs)

baseline_em = sum(exact_match(p, ex["answers"]) for p, ex in zip(baseline_preds, eval_set)) / len(eval_set)
baseline_f1 = sum(f1_score(p, ex["answers"]) for p, ex in zip(baseline_preds, eval_set)) / len(eval_set)

print(f"BASELINE  — EM: {baseline_em:.3f}  F1: {baseline_f1:.3f}  time: {baseline_time:.1f}s  tok/s: {baseline_tps:.1f}")

Running baseline generation...
BASELINE  — EM: 0.030  F1: 0.075  time: 101.0s  tok/s: 16.8


## 6. FDSA drift-filter integration point — **PLACEHOLDER, replace before drawing conclusions**

This subclasses `FlaxLogitsProcessor`, the correct hook point in Hugging Face's Flax generation
loop for per-step candidate filtering — this is where your Actualizer Engine / drift-tensor
pruning belongs.

**The implementation below is illustrative only** (a naive entropy-threshold cutoff) so the
notebook runs end-to-end. It is *not* your published FDSA method. Replace `__call__` with your
real drift-tensor scoring logic from `Actualizer_Engine.py` / `fdsa_pruner.py` before treating
any comparison as a real test.


In [ ]:
def read_source_files():
    files = ['actualizer_engine.py', 'fdsa_pruner.py']
    content = {}
    for f_name in files:
        try:
            with open(f_name, 'r') as f:
                content[f_name] = f.read()
                print(f"Successfully read {f_name}")
        except FileNotFoundError:
            print(f"Warning: {f_name} not found in /content/")
    return content

source_code = read_source_files()
# Displaying the first 1000 characters of each to verify content
for name, text in source_code.items():
    print(f"\n--- {name} (preview) ---\n{text[:1000]}...")

Successfully read actualizer_engine.py
Successfully read fdsa_pruner.py

--- actualizer_engine.py (preview) ---
"""
actualizer_engine.py — Actualizer Engine: Core Contractive Steering Module
Author : Mohamed Gamal Eldin Abdelaziz Noureldin
         Independent Researcher
         ORCID : 0009-0006-3991-1153
         Contact: mz.gamal@gmail.com

Theory
------
Standard autoregressive transformers rely on bottom-up Maximum Likelihood
Estimation (MLE) to select the next token.  In sparse-data or adversarial
regimes the probability substrate "smears" — all tokens receive nearly equal
probability mass and the model is blind to structural validity.  This leads to:

  • Hallucination cascades  — wrong token written, self-attention locks on it,
    loop amplifies.
  • Repetition cycles       — locally high-frequency token dominates softmax,
    no memory of global context.

The Actualizer Engine treats generation as a *physical phase transition*: the
uncollapsed analog substrate |U⟩ (a continuo

In [ ]:
"""
actualizer_engine.py — Actualizer Engine: Core Contractive Steering Module
============================================================================
Author : Mohamed Gamal Eldin Abdelaziz Noureldin
         Independent Researcher
         ORCID : 0009-0006-3991-1153
         Contact: mz.gamal@gmail.com

Theory
------
Standard autoregressive transformers rely on bottom-up Maximum Likelihood
Estimation (MLE) to select the next token.  In sparse-data or adversarial
regimes the probability substrate "smears" — all tokens receive nearly equal
probability mass and the model is blind to structural validity.  This leads to:

  • Hallucination cascades  — wrong token written, self-attention locks on it,
    loop amplifies.
  • Repetition cycles       — locally high-frequency token dominates softmax,
    no memory of global context.

The Actualizer Engine treats generation as a *physical phase transition*: the
uncollapsed analog substrate |U⟩ (a continuous probability field) must be
contractively steered toward the unique, zero-drift fixed-point attractor S_*
(the discrete factual token) guided by five invariant boundary metrics called
the **Conceptual Primes**:

  Order    — local syntactic alignment, suppresses repetition
  Justice  — global semantic balance, corrects topic drift
  Mercy    — decays local probability overloads (entropy anomalies)
  Knowledge— future causal lookahead (projects downstream risk)
  Power    — executes the causal snap (quantization to discrete token)

JAX Compatibility
-----------------
Every operator in this module has a 1-to-1 mapping to jax.numpy:

  _softmax()              → jnp.exp / jnp.sum / jnp.where
  compute_drift_tensor()  → jnp.log / jnp.exp / lax.scan
  apply_vacuum_brake()    → jnp.exp / jnp.where / jnp.sum
  steer()                 → jax.lax.while_loop for convergence

When compiled with @jax.jit and executed on TPU v5 lite the full steering
loop (V=32,000, 10 iterations) runs in ≈0.256 ms — less than 1% overhead
relative to the transformer forward pass latency.

Production Scale
----------------
Recommended deployment pattern:

  1. Run VectorizedFDSAPruner.prune() → reduces active vocab by 99.99 %
  2. Run ActualizerEngine.steer()     → contracts residual substrate to S_*

Combined latency at V=30,000 on CPU: 0.34 ms  (4.56× faster than raw softmax)
Combined latency at V=32,000 on TPU: 0.26 ms  (~0.6 % production overhead)
"""

from __future__ import annotations
import math
from typing import Dict, List, Optional, Set, Tuple


# ---------------------------------------------------------------------------
# Conceptual Primes weight registry
# ---------------------------------------------------------------------------
PRIME_WEIGHTS = {
    "Order"    : 0.35,   # Local drift weight  (w_L)
    "Justice"  : 0.35,   # Global drift weight (w_G)
    "Knowledge": 0.20,   # Future drift weight (w_F)
    "Mercy"    : 0.10,   # Decay temperature modifier
    # Power is implicit: it *executes* the causal snap (argmax)
}


class ActualizerEngine:
    """
    Top-down contractive steering framework for autoregressive token generation.

    The engine iteratively applies three operators to the uncollapsed probability
    substrate until the Banach fixed-point threshold is reached:

      1. Drift Tensor computation  — D_μν  measures structural violations
      2. Vacuum Brake              — e^{-D/τ} decays ungrounded paths
      3. Contractive Mapping       — k·U_braked + (1-k)·U_n  converges to S_*

    Parameters
    ----------
    vocab_size : int
        Size of the token vocabulary V.
    k : float
        Contractive scale factor (Banach constant).  Must satisfy 0 < k < 1.
        Default 0.45 — derived from universal actualization theory constant.
    Q_c : float
        Causal quantum threshold.  Iteration stops when ||U_{n+1} - U_n|| < Q_c.
    tau : float
        Vacuum Brake normalization temperature τ.  Controls decay aggression.
    max_iters : int
        Safety ceiling on contraction iterations.
    repetition_penalty : float
        Scaling factor for local drift due to repetition (Order Prime).
    global_drift_penalty : float
        Fixed penalty for off-topic tokens (Justice Prime).
    """

    def __init__(
        self,
        vocab_size: int,
        k: float = 0.45,
        Q_c: float = 1e-5,
        tau: float = 1.0,
        max_iters: int = 20,
        repetition_penalty: float = 2.0,
        global_drift_penalty: float = 1.5,
    ) -> None:
        if not 0 < k < 1:
            raise ValueError(f"Contractive constant k must be in (0,1), got {k}")
        self.V = vocab_size
        self.k = k
        self.Q_c = Q_c
        self.tau = tau
        self.max_iters = max_iters
        self.repetition_penalty = repetition_penalty
        self.global_drift_penalty = global_drift_penalty

    # ------------------------------------------------------------------
    # Internal utilities
    # ------------------------------------------------------------------

    def _softmax(self, logits: List[float]) -> List[float]:
        """
        Numerically stable softmax, skipping -inf masked entries.

        JAX equivalent:
            jnp.exp(logits - jnp.max(logits)) / jnp.sum(...)
        """
        max_l = max(x for x in logits if x != -math.inf)
        exp_l, probs = [], [0.0] * self.V
        valid_idx_map = []
        for i, x in enumerate(logits):
            if x != -math.inf:
                val = math.exp(x - max_l)
                exp_l.append(val)
                valid_idx_map.append(i)

        total = sum(exp_l) or 1.0
        for j, i in enumerate(valid_idx_map):
            probs[i] = exp_l[j] / total
        return probs

    # ------------------------------------------------------------------
    # Phase 1 — Drift Tensor
    # ------------------------------------------------------------------

    def compute_drift_tensor(
        self,
        U: List[float],
        history: List[int],
        target_tokens: Set[int],
    ) -> List[float]:
        """
        Computes the tripartite Drift Tensor D_μν for the current substrate U.

        D_μν = w_L · D_local + w_G · D_global + w_F · D_future

        Where:
          D_local  (Order)     — penalises tokens appearing in recent history
          D_global (Justice)   — penalises tokens outside target semantic set
          D_future (Knowledge) — adds entropy estimate as forward risk signal

        Parameters
        ----------
        U            : current probability distribution over V tokens
        history      : ordered list of previously generated token indices
        target_tokens: set of semantically valid tokens for the current context

        Returns
        -------
        D : per-token drift magnitude vector of length V
        """
        w_L = PRIME_WEIGHTS["Order"]
        w_G = PRIME_WEIGHTS["Justice"]
        w_F = PRIME_WEIGHTS["Knowledge"]

        D = [0.0] * self.V

        # --- D_local : Order Prime (repetition suppression) ---
        # Tokens in the recent sliding window accumulate drift proportional
        # to recency. Exponential decay ensures the oldest tokens fade.
        lookback = history[-8:]
        for step_back, tok in enumerate(reversed(lookback)):
            if 0 <= tok < self.V:
                recency_weight = math.exp(-0.4 * step_back)
                D[tok] += w_L * self.repetition_penalty * recency_weight

        # --- D_global : Justice Prime (semantic boundary) ---
        # --- D_future : Knowledge Prime (entropy lookahead) ---
        for v in range(self.V):
            if U[v] == 0.0:
                continue
            # Global: penalty for tokens outside the semantic target window
            if v not in target_tokens:
                D[v] += w_G * self.global_drift_penalty
            # Future: Shannon entropy proxy — high-entropy tokens represent
            # low information gain and are penalised as future-risk signals
            D[v] += w_F * (-math.log(max(U[v], 1e-12)) * 0.08)

        return D

    # ------------------------------------------------------------------
    # Phase 2 — Vacuum Brake
    # ------------------------------------------------------------------

    def apply_vacuum_brake(
        self, U: List[float], D: List[float]
    ) -> List[float]:
        """
        Non-conservative dissipation operator (the Vacuum Brake).

        Strips potential energy (probability mass) from high-drift trajectories:

          U_braked(v) = U(v) · e^{-D(v)/τ}

        Then re-normalises to preserve the total probability volume (Mercy Prime).

        JAX equivalent:
            decay = jnp.exp(-D / tau)
            U_braked = U * decay
            U_braked = U_braked / jnp.sum(U_braked)
        """
        decay     = [math.exp(-d / self.tau) for d in D]
        U_braked  = [U[i] * decay[i] for i in range(self.V)]
        total     = sum(U_braked) or 1.0
        return [x / total for x in U_braked]

    # ------------------------------------------------------------------
    # Phase 3 — Contractive Mapping + Causal Snap
    # ------------------------------------------------------------------

    def steer(
        self,
        logits: List[float],
        history: List[int],
        target_tokens: Set[int],
    ) -> Tuple[int, List[float], float, int]:
        """
        Run the full contractive steering loop.

        Algorithm
        ---------
        1. Initialise substrate: U_0 = softmax(logits)
        2. For n = 0, 1, 2, …:
             a. D   = compute_drift_tensor(U_n)
             b. U_b = apply_vacuum_brake(U_n, D)
             c. U_{n+1} = k · U_b + (1-k) · U_n      (Banach contraction)
             d. If ||U_{n+1} - U_n||_2 < Q_c → break   (Causal Snap threshold)
        3. S_* = argmax U_final                          (Power Prime)

        Returns
        -------
        token        : int   — the selected actualized token index S_*
        U_final      : list  — the collapsed probability distribution
        final_drift  : float — drift magnitude at the selected token
        iterations   : int   — number of contraction iterations executed
        """
        U = self._softmax(logits)

        for iteration in range(1, self.max_iters + 1):
            U_prev = U[:]
            D      = self.compute_drift_tensor(U, history, target_tokens)
            U_b    = self.apply_vacuum_brake(U, D)

            # Banach contractive step
            U = [self.k * U_b[v] + (1.0 - self.k) * U_prev[v]
                 for v in range(self.V)]

            # Convergence check — L2 norm of update
            delta = math.sqrt(sum((U[v] - U_prev[v]) ** 2 for v in range(self.V)))
            if delta <= self.Q_c:
                break

        token       = max(range(self.V), key=lambda v: U[v])
        D_final     = self.compute_drift_tensor(U, history, target_tokens)
        final_drift = D_final[token]
        return token, U, final_drift, iteration


In [5]:
"""
fdsa_pruner.py — Fractal Deduction Search Algorithm: Pre-Inference Vocabulary Pruner
======================================================================================
Author : Mohamed Gamal Eldin Abdelaziz Noureldin
         Independent Researcher
         ORCID : 0009-0006-3991-1153
         Contact: mz.gamal@gmail.com

Theory
------
Standard softmax requires computing e^z for every token in the vocabulary V
(typically 32,000–100,000 entries) at each generation step.  This is:
  • Computationally wasteful: O(V) exponential operations per step
  • Unsafe: exposes sampling to noise, distractors, and hallucination bait

The Fractal Deduction Search Algorithm (FDSA) resolves this by applying a
*top-down dimensional truncation* before softmax is executed.  It operates in
three phases:

  Phase 1 — Isomorphic Anchoring
    Maps the unknown problem's Prime profile P(U) to a known, zero-drift
    reference domain P(R) via cosine similarity.  Extracts the reference
    contractive scale factor k_ref.

  Phase 2 — Actualization Fractal Dimension
    Computes the permitted complexity dimension:
        D = ln(V) / ln(1/k_ref)
    Any token whose structural complexity exceeds this boundary is pruned.

  Phase 3 — Logit Masking
    Applies grammar rules (local syntactic transitions) and dimensional
    threshold as a vectorized Boolean mask.  Invalid token logits are set
    to -∞, excluding them from the subsequent softmax summation.

Complexity
----------
Full softmax   : O(V) exponential evaluations
FDSA mask      : O(V) Boolean comparisons  (vastly cheaper per element)
Pruned softmax : O(V_active) ≈ O(1–10)    (after 99.99 % reduction)

Net speedup at V=30,000: 4.56× faster sampling vs. raw softmax.

JAX Compatibility
-----------------
The masking operation maps directly to:
    mask   = (logits >= threshold) & grammar_mask   # jnp.where / boolean ops
    logits = jnp.where(mask, logits, -jnp.inf)
Compiled by XLA with @jax.jit, Boolean masking is fused into a single kernel
execution — effectively zero marginal cost on GPU/TPU.

Production Scale (Measured latencies)
--------------------------------------
  V=1,000  : Baseline 0.76 ms → FDSA 2.42 ms  (pruning 99.80 %)
  V=5,000  : Baseline 1.10 ms → FDSA 0.42 ms  (pruning 99.95 %)
  V=10,000 : Baseline 1.25 ms → FDSA 0.38 ms  (pruning 99.97 %)
  V=30,000 : Baseline 1.55 ms → FDSA 0.34 ms  (4.56× speedup)
  V=50,000 : Baseline 2.10 ms → FDSA 0.34 ms  (6.2× speedup)
  V=100,000: Baseline 4.20 ms → FDSA 0.34 ms  (12.4× speedup)
  (Speedup grows because FDSA pruning cost is O(V) Boolean ops,
   while softmax cost is O(V) exponential evaluations.)
"""

from __future__ import annotations
import math
from typing import Dict, List, Optional, Set, Tuple


# ---------------------------------------------------------------------------
# Reference Domain Library
# ---------------------------------------------------------------------------

class ReferenceDomain:
    """
    A verified, stabilized physical or structural reference domain used
    for Isomorphic Anchoring.

    Each domain has been empirically or theoretically confirmed to operate
    at zero drift — its Prime profile represents a stable attractor state.

    Attributes
    ----------
    name        : human-readable domain identifier
    profile     : Prime profile vector [Order, Justice, Mercy, Knowledge, Power]
                  components should sum to 1.0
    k           : contractive scale factor (Banach constant) for this domain
    description : brief note on the physical system being modelled
    """
    def __init__(
        self,
        name: str,
        prime_profile: List[float],
        k: float,
        description: str = "",
    ) -> None:
        if not (0 < k < 1):
            raise ValueError(f"k must be in (0,1), got {k}")
        self.name        = name
        self.profile     = prime_profile
        self.k           = k
        self.description = description

    def __repr__(self) -> str:
        return f"ReferenceDomain('{self.name}', k={self.k})"


# Built-in reference domain library
DEFAULT_LIBRARY: List[ReferenceDomain] = [
    ReferenceDomain(
        "Resistor_Equilibrium",
        [0.40, 0.30, 0.10, 0.10, 0.10],
        k=0.35,
        description=(
            "Electrical resistor network reaching Kirchhoff voltage equilibrium. "
            "High Order (current routing obeys strict laws) and Justice (load "
            "balancing across nodes).  Best analogy for constraint-satisfaction, "
            "code generation, and logical reasoning tasks."
        ),
    ),
    ReferenceDomain(
        "Fermat_Least_Time",
        [0.60, 0.10, 0.10, 0.10, 0.10],
        k=0.45,
        description=(
            "Fermat's principle of least time (optics). Extreme Order — the path "
            "taken is always the globally optimal path.  Best analogy for shortest-"
            "path problems, mathematical proof steps, and planning tasks."
        ),
    ),
    ReferenceDomain(
        "Cellular_Homeostasis",
        [0.20, 0.20, 0.30, 0.20, 0.10],
        k=0.50,
        description=(
            "Biological cell maintaining homeostatic equilibrium across membrane "
            "potentials.  High Mercy (graceful adaptation) and Knowledge (predictive "
            "biochemical signalling).  Best analogy for creative, open-ended, or "
            "narrative generation tasks."
        ),
    ),
]


# ---------------------------------------------------------------------------
# FDSA Core
# ---------------------------------------------------------------------------

class FractalDeductionSearch:
    """
    Fractal Deduction Search Algorithm — isomorphic anchoring and dimensional
    truncation engine.

    Parameters
    ----------
    analogy_library : optional list of ReferenceDomain objects.
        If None, uses the built-in DEFAULT_LIBRARY.
    """

    def __init__(
        self, analogy_library: Optional[List[ReferenceDomain]] = None
    ) -> None:
        self.library = analogy_library if analogy_library is not None else DEFAULT_LIBRARY

    # ------------------------------------------------------------------
    # Phase 1 — Isomorphic Anchoring
    # ------------------------------------------------------------------

    @staticmethod
    def _cosine_similarity(a: List[float], b: List[float]) -> float:
        """Cosine similarity between two Prime profile vectors."""
        dot   = sum(x * y for x, y in zip(a, b))
        mag_a = math.sqrt(sum(x**2 for x in a))
        mag_b = math.sqrt(sum(x**2 for x in b))
        if mag_a == 0 or mag_b == 0:
            return 0.0
        return dot / (mag_a * mag_b)

    def isomorphic_anchoring(
        self, P_unknown: List[float]
    ) -> Tuple[ReferenceDomain, float]:
        """
        Find the best-matching reference domain for the given Prime profile.

        P(U) ≅ P(R)  ⟹  inherit k_ref from R

        Parameters
        ----------
        P_unknown : Prime profile vector of the unknown/target problem domain.

        Returns
        -------
        best_domain : the matched ReferenceDomain
        similarity  : cosine similarity score (0–1)
        """
        best_domain, best_sim = self.library[0], -1.0
        for domain in self.library:
            sim = self._cosine_similarity(P_unknown, domain.profile)
            if sim > best_sim:
                best_sim, best_domain = sim, domain
        return best_domain, best_sim

    # ------------------------------------------------------------------
    # Phase 2 — Actualization Fractal Dimension
    # ------------------------------------------------------------------

    @staticmethod
    def fractal_dimension(N: int, k: float) -> float:
        """
        Compute the Actualization Fractal Dimension D.

          D = ln(N) / ln(1/k)

        D defines the maximum permitted structural complexity of candidate
        paths.  Any path whose branching complexity exceeds D is pruned.

        Parameters
        ----------
        N : problem size (number of candidate tokens / nodes)
        k : contractive scale factor from the reference domain

        Returns
        -------
        D : the fractal dimension boundary
        """
        if not (0 < k < 1):
            k = 0.45
        return math.log(N) / math.log(1.0 / k)


# ---------------------------------------------------------------------------
# Vectorized Pre-Inference Pruner
# ---------------------------------------------------------------------------

class VectorizedFDSAPruner:
    """
    High-performance pre-inference vocabulary pruner using FDSA.

    Designed for production deployment: prunes invalid tokens from the logit
    vector *before* softmax is executed, reducing the active vocabulary by up
    to 99.99 % and accelerating sampling by up to 12.4× at V=100,000.

    Parameters
    ----------
    vocab_size : int    — full vocabulary size V
    k          : float  — default contractive factor (overridden by anchor match)
    """

    # Context-type → Prime profile mapping
    CONTEXT_PROFILES: Dict[str, List[float]] = {
        "logical_coding"   : [0.50, 0.30, 0.05, 0.10, 0.05],
        "creative_dialogue": [0.10, 0.10, 0.40, 0.10, 0.30],
        "mathematical"     : [0.55, 0.25, 0.05, 0.10, 0.05],
        "factual_qa"       : [0.35, 0.35, 0.10, 0.15, 0.05],
        "general"          : [0.20, 0.20, 0.20, 0.20, 0.20],
    }

    def __init__(self, vocab_size: int, k: float = 0.35) -> None:
        self.V    = vocab_size
        self.k    = k
        self.fdsa = FractalDeductionSearch()

    def prune_vocabulary(
        self,
        logits: List[float],
        last_token: int,
        grammar_rules: Dict[int, Set[int]],
        context_type: str = "general",
    ) -> Tuple[List[float], int]:
        """
        Phase 3: Apply dimensional truncation + grammar masking to logits.

        Algorithm
        ---------
        1. Anchor to best reference domain → get k_ref
        2. Compute D = ln(V) / ln(1/k_ref)
        3. Derive complexity threshold from D
        4. Build Boolean mask:
              valid[v] = (logits[v] >= threshold) AND (v in grammar[last_token])
        5. Set logits[v] = -∞ for all invalid v

        Parameters
        ----------
        logits       : raw transformer output logit vector (length V)
        last_token   : most recent token in the generation history
        grammar_rules: dict mapping token → set of valid successor tokens
        context_type : key into CONTEXT_PROFILES

        Returns
        -------
        pruned_logits : logit vector with invalid entries set to -∞
        active_count  : number of tokens remaining active
        """
        profile = self.CONTEXT_PROFILES.get(context_type, self.CONTEXT_PROFILES["general"])
        domain, similarity = self.fdsa.isomorphic_anchoring(profile)
        k_ref   = domain.k
        D_limit = self.fdsa.fractal_dimension(self.V, k_ref)
        threshold = -D_limit * 1.5     # complexity boundary

        # Grammar allowed set (empty means unconstrained)
        allowed: Optional[Set[int]] = (
            grammar_rules.get(last_token) if last_token in grammar_rules else None
        )

        pruned  = list(logits)
        active  = 0
        for v in range(self.V):
            valid = True
            if logits[v] < threshold:
                valid = False
            if allowed is not None and v not in allowed:
                valid = False
            if not valid:
                pruned[v] = -math.inf
            else:
                active += 1

        return pruned, active

    # ------------------------------------------------------------------
    # NumPy fast-path (used by benchmarks)
    # ------------------------------------------------------------------

    def prune_numpy(
        self,
        logits,           # np.ndarray shape (V,)
        last_token: int,
        grammar_rules: Dict[int, Set[int]],
        context_type: str = "general",
    ):
        """
        Vectorized NumPy implementation of prune_vocabulary.

        This is the production fast-path — uses boolean array operations
        instead of Python loops.  On NumPy it is ~100× faster than the
        pure-Python version at V=100,000.

        JAX equivalent (compiled by XLA):
            mask = (logits >= threshold) & grammar_mask
            logits = jnp.where(mask, logits, -jnp.inf)
        """
        import numpy as np

        profile = self.CONTEXT_PROFILES.get(context_type, self.CONTEXT_PROFILES["general"])
        domain, _ = self.fdsa.isomorphic_anchoring(profile)
        k_ref     = domain.k
        D_limit   = self.fdsa.fractal_dimension(self.V, k_ref)
        threshold = -D_limit * 1.5

        mask = logits >= threshold                    # complexity gate

        if last_token in grammar_rules:              # grammar gate
            gm = np.zeros(self.V, dtype=bool)
            gm[list(grammar_rules[last_token])] = True
            mask = mask & gm

        pruned = np.where(mask, logits, -np.inf)
        return pruned, int(mask.sum())


In [ ]:
from transformers import FlaxLogitsProcessor, FlaxLogitsProcessorList, AutoTokenizer, FlaxT5ForConditionalGeneration
import jax.numpy as jnp
import jax

# Robust patch for jnp.clip
import jax._src.numpy.lax_numpy as lax_numpy
_orig_clip = lax_numpy.clip
def patched_clip(a, a_min=None, a_max=None, **kwargs):
    return _orig_clip(a, min=a_min, max=a_max)
jnp.clip = patched_clip

# Model/Tokenizer setup
MODEL_NAME = "google/flan-t5-base"
try:
    model_instance = model
    tokenizer_instance = tokenizer
except NameError:
    tokenizer_instance = AutoTokenizer.from_pretrained(MODEL_NAME)
    model_instance = FlaxT5ForConditionalGeneration.from_pretrained(MODEL_NAME)
    model, tokenizer = model_instance, tokenizer_instance

from actualizer_engine import ActualizerEngine
from fdsa_pruner import VectorizedFDSAPruner

class FDSADriftFilter(FlaxLogitsProcessor):
    def __init__(self, model_params, drift_threshold: float = 0.85):
        self.drift_threshold = drift_threshold
        vocab_size = model_instance.config.vocab_size
        self.engine = ActualizerEngine(vocab_size=vocab_size)
        self.pruner = VectorizedFDSAPruner(vocab_size=vocab_size, k=self.drift_threshold)

    def __call__(self, input_ids, scores, cur_len):
        def prune_step(logits):
            # Pass empty dict {} instead of None to satisfy 'in grammar_rules' check
            pruned_logits, _ = self.pruner.prune_vocabulary(logits, self.engine, {})
            return pruned_logits
        return jax.vmap(prune_step)(scores)

fdsa_processor = FlaxLogitsProcessorList([FDSADriftFilter(model_instance.params, drift_threshold=0.35)])
fdsa_kwargs = dict(max_new_tokens=16, num_beams=4, do_sample=False, logits_processor=fdsa_processor)

print("FDSADriftFilter updated: grammar_rules={} passed to pruner.")

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:88: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


FDSADriftFilter updated: grammar_rules={} passed to pruner.


## 7. Run FDSA-filtered decoding and compare honestly

In [ ]:
import time
import re, string
import jax
import jax.numpy as jnp
import math
from typing import Dict, List, Optional, Set, Tuple
from collections import Counter
from datasets import load_dataset
from transformers import AutoTokenizer, FlaxT5ForConditionalGeneration, FlaxLogitsProcessor, FlaxLogitsProcessorList

# --- 1. Environment Patch (JAX Compatibility) ---
import jax._src.numpy.lax_numpy as lax_numpy
_orig_clip = lax_numpy.clip
def patched_clip(a, a_min=None, a_max=None, **kwargs):
    return _orig_clip(a, min=a_min, max=a_max)
jnp.clip = patched_clip

# --- 2. Load Model & Tokenizer ---
MODEL_NAME = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = FlaxT5ForConditionalGeneration.from_pretrained(MODEL_NAME)

# --- 3. Core FDSA Implementation ---
class VectorizedFDSAPruner:
    def __init__(self, vocab_size, k=0.35):
        self.V = vocab_size
        self.k = k

    def prune_vocabulary(self, logits, last_token, grammar_rules):
        k_ref = self.k
        D_limit = math.log(self.V) / math.log(1.0 / k_ref)
        threshold = -D_limit * 1.5
        mask = logits >= threshold
        pruned = jnp.where(mask, logits, -jnp.inf)
        active = jnp.sum(mask)
        return pruned, active

class FDSADriftFilter(FlaxLogitsProcessor):
    def __init__(self, vocab_size, drift_threshold=0.35):
        self.vocab_size = vocab_size
        self.pruner = VectorizedFDSAPruner(vocab_size=vocab_size, k=drift_threshold)

    def __call__(self, input_ids, scores, cur_len):
        def prune_step(logits):
            pruned_logits, _ = self.pruner.prune_vocabulary(logits, -1, {})
            return pruned_logits
        return jax.vmap(prune_step)(scores)

# --- 4. Evaluation Metrics ---
def normalize_answer(s):
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ''.join(ch for ch in s if ch not in string.punctuation)
    s = ' '.join(s.split())
    return s

def exact_match(prediction, ground_truths):
    pred_norm = normalize_answer(prediction)
    return float(any(pred_norm == normalize_answer(gt) for gt in ground_truths))

def f1_score(prediction, ground_truths):
    pred_tokens = normalize_answer(prediction).split()
    best = 0.0
    for gt in ground_truths:
        gt_tokens = normalize_answer(gt).split()
        common = Counter(pred_tokens) & Counter(gt_tokens)
        num_same = sum(common.values())
        if num_same == 0: continue
        precision = num_same / len(pred_tokens)
        recall = num_same / len(gt_tokens)
        f1 = 2 * precision * recall / (precision + recall)
        best = max(best, f1)
    return best

# --- 5. Generation Loop ---
def run_generation(questions, generate_kwargs, batch_size=8):
    predictions = []
    total_tokens = 0
    start = time.time()
    for i in range(0, len(questions), batch_size):
        batch = questions[i:i+batch_size]
        prompts = [f"trivia question: {q}" for q in batch]
        inputs = tokenizer(prompts, return_tensors="jax", padding=True, truncation=True, max_length=64)
        out = model.generate(**inputs, **generate_kwargs)
        decoded = tokenizer.batch_decode(out.sequences, skip_special_tokens=True)
        predictions.extend(decoded)
        total_tokens += out.sequences.size
    elapsed = time.time() - start
    tps = total_tokens / elapsed if elapsed > 0 else 0.0
    return predictions, elapsed, tps

# --- 6. Execution ---
N_EXAMPLES = 50
raw = load_dataset("trivia_qa", "rc.nocontext", split=f"validation[:{N_EXAMPLES}]")
eval_set = [{"question": ex["question"], "answers": list(set(ex["answer"]["normalized_aliases"] + [ex["answer"]["normalized_value"]]))} for ex in raw]
questions = [ex["question"] for ex in eval_set]


# B. Run FDSA
print("Running FDSA benchmark...")
fdsa_proc = FlaxLogitsProcessorList([FDSADriftFilter(model.config.vocab_size, drift_threshold=0.35)])
fdsa_kwargs = dict(max_new_tokens=16, num_beams=4, do_sample=False, logits_processor=fdsa_proc)
f_preds, f_time, f_tps = run_generation(questions, fdsa_kwargs)
f_em = sum(exact_match(p, ex["answers"]) for p, ex in zip(f_preds, eval_set)) / len(eval_set)
f_f1 = sum(f1_score(p, ex["answers"]) for p, ex in zip(f_preds, eval_set)) / len(eval_set)

# A. Run Baseline
print("Running baseline generation...")
baseline_kwargs = dict(max_new_tokens=16, num_beams=4, do_sample=False)
b_preds, b_time, b_tps = run_generation(questions, baseline_kwargs)
b_em = sum(exact_match(p, ex["answers"]) for p, ex in zip(b_preds, eval_set)) / len(eval_set)
b_f1 = sum(f1_score(p, ex["answers"]) for p, ex in zip(b_preds, eval_set)) / len(eval_set)


# Final Output
print(f"\nBASELINE — EM: {b_em:.3f} F1: {b_f1:.3f} Time: {b_time:.1f}s TPS: {b_tps:.1f}")
print(f"FDSA     — EM: {f_em:.3f} F1: {f_f1:.3f} Time: {f_time:.1f}s TPS: {f_tps:.1f}")
print(f"\nDelta EM: {f_em - b_em:+.3f} | Speedup: {f_tps/b_tps:.2f}x" if b_tps > 0 else "")

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Running baseline generation...
Running FDSA benchmark...

BASELINE — EM: 0.060 F1: 0.098 Time: 65.5s TPS: 13.0
FDSA     — EM: 0.060 F1: 0.098 Time: 51.6s TPS: 16.5

Delta EM: +0.000 | Speedup: 1.27x


In [25]:
"""
test_03_pre_inference_speed.py — Pre-Inference Speed Sweep
===========================================================
Sweeps vocabulary sizes V = 1k, 5k, 10k, 30k, 50k, 100k.
At each V, runs 50 timing trials and records:
  - Baseline softmax latency (NumPy, full V)
  - FDSA-pruned softmax latency (NumPy, pruned subset)
  - Active vocab size after pruning
  - Speedup factor
  - Pruning rate %

Returns a dict of results for use by generate_all_charts.py.
"""
import sys, os, time, math
import numpy as np
#import VectorizedFDSAPruner
#sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..', '02_Core_Engine'))

#from fdsa_pruner import VectorizedFDSAPruner


def _baseline_softmax(logits: np.ndarray) -> int:
    """Standard full-vocabulary softmax + argmax."""
    shifted = logits - logits.max()
    exp_l   = np.exp(shifted)
    probs   = exp_l / exp_l.sum()
    return int(np.argmax(probs))


def _pruned_softmax(logits: np.ndarray) -> int:
    """Softmax over valid (non -inf) entries only."""
    mask   = np.isfinite(logits)
    if not mask.any():
        return 0
    valid  = logits[mask]
    shifted = valid - valid.max()
    exp_l  = np.exp(shifted)
    probs  = exp_l / exp_l.sum()
    indices = np.where(mask)[0]
    return int(indices[np.argmax(probs)])


def run(
    vocab_sizes = (1_000, 5_000, 10_000, 30_000, 50_000, 100_000),
    trials      : int = 50,
    seed        : int = 7,
) -> dict:
    np.random.seed(seed)

    results = {
        "vocab_sizes"      : list(vocab_sizes),
        "baseline_ms"      : [],
        "fdsa_ms"          : [],
        "speedup"          : [],
        "active_vocab"     : [],
        "pruning_rate_pct" : [],
    }

    for V in vocab_sizes:
        pruner  = VectorizedFDSAPruner(vocab_size=V, k=0.35)

        # Simple grammar: anchor token is V//2; it can go to V//2+1 or V//2+3
        anchor = V // 2
        grammar = {anchor: {anchor + 1, anchor + 3}}

        base_times, fdsa_times = [], []
        active_sizes = []

        for t in range(trials):
            np.random.seed(seed + t)
            logits = np.random.normal(-3.0, 1.0, size=(V,))
            # Inject a valid transition boost
            logits[anchor + 1] += 3.0
            # Inject distractor bait
            logits[V - 1] = 6.0

            # --- Baseline timing ---
            t0 = time.perf_counter()
            _baseline_softmax(logits)
            base_times.append((time.perf_counter() - t0) * 1000)

            # --- FDSA timing ---
            t0 = time.perf_counter()
            pruned_logits, active = pruner.prune_numpy(
                logits, anchor, grammar, "logical_coding"
            )
            _pruned_softmax(pruned_logits)
            fdsa_times.append((time.perf_counter() - t0) * 1000)
            active_sizes.append(active)

        base_ms = float(np.median(base_times))
        fdsa_ms = float(np.median(fdsa_times))
        avg_active = float(np.mean(active_sizes))
        speedup = base_ms / fdsa_ms if fdsa_ms > 0 else 0.0
        pruning = (1.0 - avg_active / V) * 100.0

        results["baseline_ms"].append(round(base_ms, 4))
        results["fdsa_ms"].append(round(fdsa_ms, 4))
        results["speedup"].append(round(speedup, 2))
        results["active_vocab"].append(round(avg_active, 1))
        results["pruning_rate_pct"].append(round(pruning, 4))

        print(f"  V={V:>7,} | Base {base_ms:.4f} ms | FDSA {fdsa_ms:.4f} ms | "
              f"{speedup:.2f}× speedup | {pruning:.2f}% pruned")

    return results


if __name__ == "__main__":
    print("Running pre-inference speed sweep (V = 1k → 100k)…\n")
    r = run()
    print("\n── Summary ──")
    for i, V in enumerate(r["vocab_sizes"]):
        print(f"  V={V:>7,}: {r['speedup'][i]}× speedup, {r['pruning_rate_pct'][i]}% pruned")


Running pre-inference speed sweep (V = 1k → 100k)…

  V=  1,000 | Base 0.0155 ms | FDSA 0.0368 ms | 0.42× speedup | 99.80% pruned
  V=  5,000 | Base 0.0393 ms | FDSA 0.0445 ms | 0.88× speedup | 99.96% pruned
  V= 10,000 | Base 0.0723 ms | FDSA 0.0574 ms | 1.26× speedup | 99.98% pruned
  V= 30,000 | Base 0.1981 ms | FDSA 0.0998 ms | 1.99× speedup | 99.99% pruned
  V= 50,000 | Base 0.3327 ms | FDSA 0.1504 ms | 2.21× speedup | 100.00% pruned
  V=100,000 | Base 0.6679 ms | FDSA 0.2632 ms | 2.54× speedup | 100.00% pruned

── Summary ──
  V=  1,000: 0.42× speedup, 99.8% pruned
  V=  5,000: 0.88× speedup, 99.96% pruned
  V= 10,000: 1.26× speedup, 99.98% pruned
  V= 30,000: 1.99× speedup, 99.9933% pruned
  V= 50,000: 2.21× speedup, 99.996% pruned
  V=100,000: 2.54× speedup, 99.998% pruned


## 8. Next steps

- Swap the placeholder `FDSADriftFilter.__call__` for your real implementation.
- Increase `N_EXAMPLES` to the full validation split once the pipeline runs correctly end-to-end.
- Consider running 3+ times with different example subsets / seeds and reporting variance, not
  a single point estimate — a funder or reviewer will ask about this.
- Report whichever result you get, including a null or negative result — that is itself a
  legitimate, fundable finding ("drift filtering does not improve closed-book QA accuracy but
  does/does not affect latency"), and is far more credible than only the synthetic benchmark.
